In [ ]:
#NoteBooK para fazer o data aug de tudo com base em todas as funções que foram criada até agora

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import random
import os
from tqdm import tqdm # Biblioteca para barra de progresso

In [ ]:
# --- Parâmetros ---
csv_file = "mitbih_all_records_renumerada.csv"
df_main = pd.read_csv(csv_file)

In [ ]:
def plot_beat_segment(segment_df, title="Segmento de Batimento Cardíaco"):
    """Função auxiliar para plotar um segmento de batimento de um DataFrame."""
    if segment_df is None or segment_df.empty:
        print("DataFrame vazio. Nada para plotar.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(segment_df["sample #"], segment_df["amplitude"], label="Sinal")
    plt.title(title)
    plt.xlabel("Número da Amostra")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [ ]:
def save_augmented_segment(segment_df, output_dir, file_name):
    """
    Salva um único segmento em CSV, garantindo o formato correto das colunas.
    """
    # Garante que o DataFrame tenha as colunas no formato original para salvar
    augmented_df = segment_df.copy()
    
    if "amplitude" in augmented_df.columns:
        augmented_df["channel_0"] = augmented_df["amplitude"]
    
    # Garante que as colunas essenciais existam
    if "type" not in augmented_df.columns: augmented_df['type'] = 'unknown'
    if "sample #" not in augmented_df.columns: augmented_df['sample #'] = np.arange(len(augmented_df))

    # Seleciona e ordena as colunas
    augmented_df = augmented_df[["channel_0", "sample #", "type"]]

    if (file_name == "/"):
        file_name = "B"
    
    # Constrói o caminho completo do arquivo e salva
    full_path = os.path.join(output_dir, file_name)
    augmented_df.to_csv(full_path, index=False)

In [ ]:

def save_augmented(segment_df, output_dir, file_name):
    """Salva segmento em CSV, construindo o caminho completo."""
    # Garante que o DataFrame tenha as colunas no formato original para salvar
    augmented_df = segment_df.copy()
    
    if "amplitude" in augmented_df.columns:
        augmented_df["channel_0"] = augmented_df["amplitude"]
    
    # Seleciona e ordena as colunas
    if "type" not in augmented_df.columns: augmented_df['type'] = 'unknown' # Garante que a coluna 'type' exista
    if "sample #" not in augmented_df.columns: augmented_df['sample #'] = np.arange(len(augmented_df)) # Garante que a coluna 'sample #' exista

    augmented_df = augmented_df[["channel_0", "sample #", "type"]]

    if (file_name == "/"):
        file_name = "B"
    
    # Constrói o caminho completo do arquivo
    full_path = os.path.join(output_dir, file_name)
    augmented_df.to_csv(full_path, index=False)

In [ ]:
def augment_time_shift(segment_df, shift_fraction=0.05, fill_value=0.0):
    """
    Desloca o sinal de ECG no tempo (para esquerda ou direita) por uma fração
    aleatória do comprimento do segmento. O espaço vazio é preenchido com `fill_value`.

    Parâmetros:
    - segment_df (pd.DataFrame): DataFrame com a coluna 'amplitude'.
    - shift_fraction (float): Fração máxima do comprimento total para o deslocamento (ex: 0.05 para 5%).
    - fill_value (float): Valor para preencher os espaços vazios criados pelo deslocamento.

    Retorna:
    - pd.DataFrame: O DataFrame com o sinal deslocado no tempo.
    """
    augmented_df = segment_df.copy()
    y = segment_df["amplitude"].values
    
    # Calcula o limite máximo de deslocamento em número de amostras
    max_shift_samples = int(len(y) * shift_fraction)
    
    # Sorteia um deslocamento aleatório (pode ser positivo/direita ou negativo/esquerda)
    shift_samples = random.randint(-max_shift_samples, max_shift_samples)
    
    # Cria um novo array preenchido com o valor de preenchimento
    shifted_y = np.full_like(y, fill_value)
    
    # Aplica o deslocamento copiando os dados
    if shift_samples > 0:
        # Deslocamento para a direita: copia o início de y para o final de shifted_y
        shifted_y[shift_samples:] = y[:-shift_samples]
    elif shift_samples < 0:
        # Deslocamento para a esquerda: copia o final de y para o início de shifted_y
        shifted_y[:shift_samples] = y[-shift_samples:]
    else:
        # Sem deslocamento
        shifted_y = y
        
    augmented_df["amplitude"] = shifted_y
    
    return augmented_df

In [ ]:
def get_beat_interval_robust_optimized(df, all_peaks, target_rows, nth=0, channel="channel_0"):
    """
    Versão otimizada que recebe um DataFrame e picos pré-calculados para extrair
    o intervalo R-R de um batimento específico.
    """
    if nth >= len(target_rows):
        return None

    center_sample = int(target_rows.iloc[nth]["sample #"])

    # Encontra o índice do pico mais próximo da anotação
    center_peak_index = np.argmin(np.abs(all_peaks - center_sample))
    
    # Verificação de borda
    if center_peak_index == 0 or center_peak_index >= len(all_peaks) - 1:
        return None

    # Pega os picos vizinhos
    start_peak = all_peaks[center_peak_index - 1]
    end_peak = all_peaks[center_peak_index + 1]
    
    # Extrai o segmento
    mask = (df["sample #"] >= start_peak) & (df["sample #"] <= end_peak)
    segment_df = df.loc[mask].copy()

    # Renomeia a coluna para o padrão "amplitude"
    if channel in segment_df.columns:
        segment_df.rename(columns={channel: "amplitude"}, inplace=True)
    
    return segment_df

In [ ]:

def augment_add_sine_pulse(segment_df, amplitude=0.1, frequency=1.0, random_phase=True):
    augmented_df = segment_df.copy()
    y = segment_df["amplitude"].values
    x = np.arange(len(y))
    phase = np.random.uniform(0, 2 * np.pi) if random_phase else 0
    sine_wave = amplitude * np.sin(2 * np.pi * frequency * (x / len(x)) + phase)
    augmented_df["amplitude"] = y + sine_wave
    return augmented_df

In [ ]:

def augment_jitter_2(segment_df, save_dir, file_name, sigma_factor=0.02, variable_sigma=True, seed=None):
    rng = np.random.default_rng(seed)
    augmented_df = segment_df.copy()
    base_sigma = sigma_factor * np.std(segment_df["amplitude"])
    
    if variable_sigma:
        sigma_series = base_sigma * rng.lognormal(mean=0, sigma=0.25, size=len(segment_df))
    else:
        sigma_series = np.full(len(segment_df), base_sigma)
        
    noise = rng.normal(0, sigma_series)
    augmented_df["amplitude"] += noise
    
    # Salva o resultado chamando a função auxiliar
    save_augmented(augmented_df, save_dir, file_name)
    
    return augmented_df

In [ ]:
# Função para adicionar ruído (Jitter)
def augment_jitter(segment_df, sigma_factor=0.02, variable_sigma=True, seed=None):
    rng = np.random.default_rng(seed)
    augmented_df = segment_df.copy()
    base_sigma = sigma_factor * np.std(segment_df["amplitude"])
    
    if variable_sigma:
        sigma_series = base_sigma * rng.lognormal(mean=0, sigma=0.25, size=len(segment_df))
    else:
        sigma_series = np.full(len(segment_df), base_sigma)
        
    noise = rng.normal(0, sigma_series)
    augmented_df["amplitude"] += noise
    return augmented_df

In [ ]:
# --- NOVA FUNÇÃO COMBINADA ---
def augment_sine_and_jitter(segment_df, sine_amplitude=0.1, sine_frequency=1.0, sigma_factor=0.015):
    """
    Aplica uma sequência de aumentos: primeiro SinePulse e depois Jitter.
    """
    # Passo 1: Adicionar a deriva da linha de base
    sine_df = augment_add_sine_pulse(segment_df, amplitude=sine_amplitude, frequency=sine_frequency)
    
    # Passo 2: Adicionar ruído no resultado da primeira transformação
    final_augmented_df = augment_jitter(sine_df, sigma_factor=sigma_factor)
    
    return final_augmented_df

In [ ]:
def augment_amplitude_scale(segment_df, scale_range=(0.8, 1.2)):
    """
    Aplica uma escala na amplitude do sinal, multiplicando todos os valores
    por um fator aleatório dentro do intervalo especificado.

    Parâmetros:
    - segment_df (pd.DataFrame): DataFrame com a coluna 'amplitude'.
    - scale_range (tuple): Tupla (min, max) para o fator de escala aleatório.

    Retorna:
    - pd.DataFrame: O DataFrame com a amplitude escalada.
    """
    augmented_df = segment_df.copy()
    
    # Sorteia um fator de escala aleatório do intervalo
    scale_factor = random.uniform(scale_range[0], scale_range[1])
    
    # Aplica a multiplicação na coluna de amplitude
    augmented_df["amplitude"] = segment_df["amplitude"] * scale_factor
    
    return augmented_df

In [ ]:
def augment_scale_and_jitter(segment_df, scale_range=(0.9, 1.1), sigma_factor=0.02):
    """
    Aplica uma sequência de aumentos de dados: primeiro AmplitudeScale e depois Jitter.

    Parâmetros:
    - segment_df (pd.DataFrame): O DataFrame do segmento original.
    - scale_range (tuple): O intervalo para o AmplitudeScale.
    - sigma_factor (float): O fator de sigma para o Jitter.

    Retorna:
    - pd.DataFrame: O DataFrame com as duas transformações aplicadas.
    """
    #  Aplicar a escala de amplitude
    scaled_df = augment_amplitude_scale(segment_df, scale_range=scale_range)
    
    # Aplicar o jitter no resultado da escala
    final_augmented_df = augment_jitter(scaled_df, sigma_factor=sigma_factor)
    
    return final_augmented_df

In [ ]:
def augment_amplitude_scale(segment_df, scale_range=(0.8, 1.2)):
    """
    Aplica uma escala na amplitude do sinal, multiplicando todos os valores
    por um fator aleatório dentro do intervalo especificado.

    Parâmetros:
    - segment_df (pd.DataFrame): DataFrame com a coluna 'amplitude'.
    - scale_range (tuple): Tupla (min, max) para o fator de escala aleatório.

    Retorna:
    - pd.DataFrame: O DataFrame com a amplitude escalada.
    """
    augmented_df = segment_df.copy()
    
    # Sorteia um fator de escala aleatório do intervalo
    scale_factor = random.uniform(scale_range[0], scale_range[1])
    
    # Aplica a multiplicação na coluna de amplitude
    augmented_df["amplitude"] = segment_df["amplitude"] * scale_factor
    
    return augmented_df

In [ ]:
def augment_shift_and_jitter(segment_df, shift_fraction=0.05, sigma_factor=0.015, fill_value=0.0):
    """
    Aplica uma sequência de aumentos de dados: primeiro TimeShift e depois Jitter.

    Parâmetros:
    - segment_df (pd.DataFrame): O DataFrame do segmento original.
    - shift_fraction (float): A fração máxima para o deslocamento no tempo.
    - sigma_factor (float): O fator de sigma para o Jitter.
    - fill_value (float): O valor para preencher as bordas no TimeShift.

    Retorna:
    - pd.DataFrame: O DataFrame com as duas transformações aplicadas.
    """
    # Passo 1: Aplicar o deslocamento no tempo
    shifted_df = augment_time_shift(segment_df, shift_fraction=shift_fraction, fill_value=fill_value)
    
    # Passo 2: Adicionar ruído (jitter) no resultado do deslocamento
    final_augmented_df = augment_jitter(shifted_df, sigma_factor=sigma_factor)
    
    return final_augmented_df

In [ ]:
def augment_time_shift(segment_df, shift_fraction=0.05, fill_value=0.0):
    """
    Desloca o sinal de ECG no tempo (para esquerda ou direita) por uma fração
    aleatória do comprimento do segmento. O espaço vazio é preenchido com `fill_value`.

    Parâmetros:
    - segment_df (pd.DataFrame): DataFrame com a coluna 'amplitude'.
    - shift_fraction (float): Fração máxima do comprimento total para o deslocamento (ex: 0.05 para 5%).
    - fill_value (float): Valor para preencher os espaços vazios criados pelo deslocamento.

    Retorna:
    - pd.DataFrame: O DataFrame com o sinal deslocado no tempo.
    """
    augmented_df = segment_df.copy()
    y = segment_df["amplitude"].values
    
    # Calcula o limite máximo de deslocamento em número de amostras
    max_shift_samples = int(len(y) * shift_fraction)
    
    # Sorteia um deslocamento aleatório (pode ser positivo/direita ou negativo/esquerda)
    shift_samples = random.randint(-max_shift_samples, max_shift_samples)
    
    # Cria um novo array preenchido com o valor de preenchimento
    shifted_y = np.full_like(y, fill_value)
    
    # Aplica o deslocamento copiando os dados
    if shift_samples > 0:
        # Deslocamento para a direita: copia o início de y para o final de shifted_y
        shifted_y[shift_samples:] = y[:-shift_samples]
    elif shift_samples < 0:
        # Deslocamento para a esquerda: copia o final de y para o início de shifted_y
        shifted_y[:shift_samples] = y[-shift_samples:]
    else:
        # Sem deslocamento
        shifted_y = y
        
    augmented_df["amplitude"] = shifted_y
    
    return augmented_df

In [ ]:
def time_shift_jitter_8000(num_augmentations, target_type, df_main, count):
    output_dir = "data_aug_shift_jitter_multiple_files/" # MUDANÇA AQUI

    
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    # --- Loop de Aumento e Salvamento ---
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # MUDANÇA AQUI: Chama a nova função combinada
            augmented_df = augment_shift_and_jitter(segment_df, shift_fraction=0.05, sigma_factor=0.015)

            if (target_type == "/"):
                target_type = "B"
            
            # MUDANÇA AQUI: Define o novo nome do arquivo
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_shift_jitter.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
            
    

In [ ]:
def amplitude_scale_jitter(num_augmentations, target_type, df_main, count):
    
    output_dir = "data_amplitude_scale_jitter_multiple_files/"

    
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    # --- Loop de Aumento e Salvamento ---
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # Chama a nova função combinada
            augmented_df = augment_scale_and_jitter(segment_df, scale_range=(0.9, 1.1), sigma_factor=0.015)

            if (target_type == "/"):
                target_type = "B"
            
            # Define o novo nome do arquivo
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_scale_jitter.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
            


In [ ]:
def aug_jitter(num_augmentations, target_type, df_main, count):
   
    output_dir = "data_aug/"
    
    
    # =================================================================
    # PASSO 2: EXECUTAR O LOOP RÁPIDO, SALVANDO A CADA ITERAÇÃO
    # =================================================================
    
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    for i in range(num_augmentations):
        # Pega o segmento rapidamente
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None:

            if (target_type == "/"):
                target_type = "B"
            
            # Define um nome de arquivo único para esta iteração
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_jitter.csv"
            
            # Aplica o Jitter e salva o arquivo
            augment_jitter_2(
                segment_df=segment_df,
                save_dir=output_dir,
                file_name=file_name
            )
            
   

In [ ]:
def aug_sine(num_augmentations, target_type, df_main, count):

    output_dir = "data_aug_sine_multiple_files/"
    
    # --- Loop de Aumento e Salvamento ---

    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # Randomiza os parâmetros da augmentation
            amp = random.uniform(0.05, 0.1)
            freq = random.uniform(0.5, 2.0)
            
            # Aplica a augmentation
            augmented_df = augment_add_sine_pulse(segment_df, amplitude=amp, frequency=freq)

            if (target_type == "/"):
                target_type = "B"
            
            # Define nome do arquivo e salva
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_sine.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
            
   

In [ ]:
def aug_jitter_sine(num_augmentations, target_type, df_main, count):

    output_dir = "data_aug_sine_jitter_L_multiple_files/" # MUDANÇA AQUI

    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25

    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    # --- Loop de Aumento e Salvamento ---
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # MUDANÇA AQUI: Randomiza parâmetros e chama a nova função combinada
            sine_amp = random.uniform(0.05, 0.1)
            sine_freq = random.uniform(0.5, 2.0)
            sigma_fact = random.uniform(0.010, 0.015)
            augmented_df = augment_sine_and_jitter(
                segment_df, 
                sine_amplitude=sine_amp, 
                sine_frequency=sine_freq,
                sigma_factor=sigma_fact
            )

            if (target_type == "/"):
                target_type = "B"
                
            # MUDANÇA AQUI: Define o novo nome do arquivo
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_sine_jitter.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
        

In [ ]:
def time_shift(num_augmentations, target_type, df_main,count):
    output_dir = "data_time_shift_multiple_files/"
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25

    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    # --- Loop de Aumento e Salvamento ---
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:

            shift_fract = random.uniform(0.05, 0.10)
            
            # Aplica a nova augmentation de deslocamento no tempo
            augmented_df = augment_time_shift(segment_df, shift_fraction= shift_fract)
            if (target_type == "/"):
                target_type = "B"
            
            # Define nome do arquivo e salva
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_shift.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)
        

In [ ]:
def amplidude_scale(num_augmentations, target_type, df_main, count):
    output_dir = "F/"
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25

    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    # --- Loop de Aumento e Salvamento ---
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            # Aplica a nova augmentation de escala de amplitude
            augmented_df = augment_amplitude_scale(segment_df, scale_range=(0.8, 1.2))

            if (target_type == "/"):
                target_type = "B"
            
            # Define nome do arquivo e salva
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_aug_scale.csv"
            save_augmented_segment(augmented_df, output_dir, file_name)

In [ ]:
def original_beat(num_augmentations, target_type, df_main, count):
    output_dir = "N/"
    
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25

    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    # --- Loop de Aumento e Salvamento ---
    for i in range(num_augmentations):
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None and not segment_df.empty:
            if (target_type == "/"):
                target_type = "B"
            
            # Define nome do arquivo e salva
            file_name = f"{target_type}_beat_{i+(num_augmentations*count)}_original.csv"
            save_augmented_segment(segment_df, output_dir, file_name)

In [ ]:
def pre_calculus(csv_file, target_type, num_augmentations):
    # Parâmetros da detecção de picos
    peak_distance = 180
    peak_height = 0.25
    
    # --- Pré-cálculo ---
    print("1/3 - Carregando arquivo CSV...")
    df_main = pd.read_csv(csv_file)
    
    print(f"2/3 - Encontrando anotações '{target_type}'...")
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        num_augmentations = len(target_rows)
        print(f"Aviso: Reduzindo para {num_augmentations} aumentos, pois é o total de batimentos encontrados.")

    print("3/3 - Detectando picos R...")
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=peak_distance, height=peak_height)
    return peaks


In [ ]:
target_type = "N"
num_augmentations = 10000

In [ ]:
#numeros de classes total (aug + original) a serem trabalhadas (numero de aug)
#A = 6000(3454)
#B = 8000(972)
#f = 4000(3018)
#F = 4000(3197)
#L = 8075(0)
#N = 10000(0)
#R = 8000(741)
#V = 8000(870)


In [ ]:
peaks= pre_calculus(csv_file, target_type,num_augmentations)

In [ ]:
for i in tqdm(range(1,37526), "Carregando"):
    time_shift(num_augmentations, target_type, df_main, i)

In [ ]:
for i in tqdm(range(1,37526), "Carregando"):
    aug_jitter_sine(num_augmentations, target_type, df_main, i)

In [ ]:
for i in tqdm(range(1,37526), "Carregando"):
    aug_sine(num_augmentations, target_type, df_main, i)

In [ ]:
for i in tqdm(range(1,37526), "Carregando"):
    aug_jitter(num_augmentations, target_type, df_main, i)

In [ ]:
for i in tqdm(range(1,37526), "Carregando"):
    amplitude_scale_jitter(num_augmentations, target_type, df_main, i)

In [ ]:
for i in tqdm(range(1,37526), "Carregando"):
    time_shift_jitter_8000(num_augmentations, target_type, df_main, i)

In [ ]:
for i in tqdm(range(0,1), "Carregando"):
    original_beat(num_augmentations, target_type, df_main, i)